In [ ]:
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
import sys
sys.path.append("../")
import models
from models import baselines
from utils.training import train_meta_model
from utils.data_utils import ctxt_trgt_split, obtain_me_a_nice_gp_dataset_please

torch.set_default_dtype(torch.float32)

%load_ext autoreload
%autoreload 2

### Make Dataset

In [ ]:
num_datasets = 10_000
md = []
# gp_data_hypers = {'l': 1.0, 'kernel': 'per', 'p': 1, 'x_range': [-5.0, 5.0]}
gp_data_hypers = {'l': 0.5, 'kernel': 'se', 'x_range': [-4.0, 4.0]}
# st_data_hypers = {'p': 1.0, 'random_shift': False, 'random_gradient': False, 'x_range': [-5.0, 5.0]}
# h_data_hypers = {'x_range': [-5.0, 5.0], 'l': 1}
for _ in range(num_datasets):
    X, y = obtain_me_a_nice_gp_dataset_please(n_range=[25, 50], **gp_data_hypers)
    # X, y = obtain_me_a_nice_sawtooth_dataset_please(n_range=[40, 100], **st_data_hypers)
    # X, y = obtain_me_a_nice_heaviside_dataset_please(n_range=[40, 100], **h_data_hypers)
    md.append((X, y))

### Initialise Model

In [ ]:
lik = models.GaussianLikelihood(y_dim=1, sigma_y=0.5, train=True)
# np = models.BDNP(x_dim=1,
#                  y_dim=1,
#                  hidden_dims=[32,32],
#                  prior_type=1,
#                  likelihood=lik,
#                  inf_dims=[32,32],
#                  use_final_layer_targets=True,
#                  scale_prior=True,
#                  nonlinearity=torch.nn.SiLU(),
#                  )
# np.set_prior_trainability(0.5, from_front=False)
# np = baselines.NP(x_dim=1,
#                     y_dim=1,
#                     lik=lik,
#                     encoder_dims=[256, 256, 256],
#                     decoder_dims=[256, 256],
#                     nonlinearity=torch.nn.ReLU(),
#                     )

# np = baselines.CNP(x_dim=1,
#                    y_dim=1,
#                    encoder_dims=[64, 64],
#                    decoder_dims=[64, 64],
#                    nonlinearity=torch.nn.ReLU(),
#                    )

# np = baselines.TNP(x_dim=1, y_dim=1, num_layers=2, r_dim=64, nonlinearity=torch.nn.ReLU())
# np = baselines.EQTNP(x_dim=1, y_dim=1, num_blocks=2)
np = baselines.EpiTNP(x_dim=1, y_dim=1, lik=lik, num_blocks=3)

### Train the Model

In [ ]:
training_metrics = train_meta_model(
    np,
    md,
    training_steps=20_000,
    batch_size=5,
    learning_rate=3e-3,
    final_learning_rate=3e-4,
    num_samples=16,
    loss_function='mpl',
    ctxt_proportion_range=[0.1, 0.5]
)
fig, axes = plt.subplots(1, len(training_metrics), figsize=(3*len(training_metrics), 1))
omitted_steps = 100
if len(training_metrics) == 1:
    for key, value in training_metrics.items():
        axes.plot(value[omitted_steps:])
        axes.set_xlabel(key)
        axes.grid()
else:
    for i, (key, value) in enumerate(training_metrics.items()):
        axes[i].plot(value[omitted_steps:])
        axes[i].set_xlabel(key)
        axes[i].grid()
    # axes[i].set_ylim([-100, 400])
plt.show()

### Visualise test predictions

In [ ]:
prior_samps = False
# prior_samps = True
ar = False
ar = True

if prior_samps:
    X_c, y_c = None, None
    samps = 100
else:
    # X, y = obtain_me_a_nice_gp_dataset_please(n_range=[10, 100], **gp_data_hypers)
    gp_data_hypers['x_range'] = [-2.0, 2.0]
    X, y = obtain_me_a_nice_gp_dataset_please(n_range=[1, 40], **gp_data_hypers)
    # X, y = obtain_me_a_nice_sawtooth_dataset_please(n_range=[10, 100], **st_data_hypers)
    # X, y = obtain_me_a_nice_heaviside_dataset_please(n_range=[10, 100], **h_data_hypers)
    # X, y = md[0]
    X_c, y_c = X.clone(), y.clone()
    samps = 100


xs = torch.linspace(-2.5, 2.5, 200).unsqueeze(-1)#.to(dtype=torch.float32)
with torch.no_grad():
    if isinstance(np, baselines.CNP) or isinstance(np, baselines.TNP) or isinstance(np, baselines.EQTNP):
        if isinstance(np, baselines.TNP) and ar:
            preds = np.autoregressive_forward(xs, X_c, y_c, num_samples=samps, verbose=True)
        else:
            preds = np(xs, X_c, y_c)
    else:
        preds = np(xs, X_c, y_c, num_samples=samps).cpu()
if isinstance(preds, torch.distributions.Distribution):
    m, std = preds.mean, preds.variance.sqrt()
    if len(m.shape) == 2:
        m, std = m.squeeze(-1), std.squeeze(-1)
    plt.plot(xs, m, linewidth=1.0, color='C0')
    plt.fill_between(xs.squeeze(), m+2*std, m-2*std, color='C0', alpha=0.5)
else:
    plt.plot(xs.unsqueeze(0).repeat((samps, 1, 1)).squeeze(-1).T, preds.squeeze(-1).T, linewidth=0.5, color='C0', alpha=0.5)
if X_c is not None:
    plt.scatter(X_c, y_c, color='C1', zorder=10000)
plt.grid()
plt.xlim([-2.5, 2.5])
plt.ylim([-5.0, 5.0])
plt.show()

In [ ]:
raise ValueError("stop code here.")

### Online Learning Demo

In [ ]:
X, y = obtain_me_a_nice_gp_dataset_please(n_range=[30, 31], **gp_data_hypers)
# X, y = obtain_me_a_nice_sawtooth_dataset_please(n_range=[30, 32], **st_data_hypers)
# X, y = obtain_me_a_nice_heaviside_dataset_please(n_range=[30, 32], **h_data_hypers)
X_c, y_c = X.clone(), y.clone()
samps = 100
inds = [0, 3, 8, 15, 30]

fig, axes = plt.subplots(4, 3, sharex=True, sharey=True, figsize=(9, 12))

xs = torch.linspace(-5.0, 5.0, 200).unsqueeze(-1)

for j in range(1, len(inds)):

    with torch.no_grad():
        full_pred_samps = bdnp(xs, X_c[:inds[j]], y_c[:inds[j]], num_samples=samps)[0]
        seq_pred_samps = bdnp(xs, X_c[inds[j-1]:inds[j]], y_c[inds[j-1]:inds[j]], num_samples=samps, update_prev=(j!=1), save_stuff=True)[0]
        control_pred_samps = bdnp(xs, X_c[inds[j-1]:inds[j]], y_c[inds[j-1]:inds[j]], num_samples=samps)[0]

    axes[j-1][0].plot(xs.unsqueeze(0).repeat((samps, 1, 1)).squeeze(-1).T, full_pred_samps.squeeze(-1).T, linewidth=0.5, color='C0', alpha=0.5)
    axes[j-1][0].scatter(X_c[:inds[j]], y_c[:inds[j]], color='C1', zorder=1000)

    axes[j-1][1].plot(xs.unsqueeze(0).repeat((samps, 1, 1)).squeeze(-1).T, seq_pred_samps.squeeze(-1).T, linewidth=0.5, color='C0', alpha=0.5)
    axes[j-1][1].scatter(X_c[inds[j-1]:inds[j]], y_c[inds[j-1]:inds[j]], color='C1', zorder=1000)
    axes[j-1][1].scatter(X_c[:inds[j-1]], y_c[:inds[j-1]], color='black', zorder=1000)

    axes[j-1][2].plot(xs.unsqueeze(0).repeat((samps, 1, 1)).squeeze(-1).T, control_pred_samps.squeeze(-1).T, linewidth=0.5, color='C0', alpha=0.5)
    axes[j-1][2].scatter(X_c[inds[j-1]:inds[j]], y_c[inds[j-1]:inds[j]], color='C1', zorder=1000)

    for k in range(3):
        axes[j-1][k].grid()
        axes[j-1][k].set_xlim([-5.0, 5.0])
        axes[j-1][k].set_ylim([-5.0, 5.0])

plt.show()